# Delta Lake com Apache Spark

## Cenário: Loja Virtual

Este notebook demonstra o uso do **Delta Lake** integrado ao **Apache Spark (PySpark)** para gerenciar dados de uma **Loja Virtual**.

---

## Modelo de Dados (ER)

![Diagrama Entidade-Relacionamento](../docs/assets/er_diagram.png)

## Fonte de Dados

Dados **sintéticos** gerados diretamente no notebook, representando uma loja virtual com clientes, produtos e pedidos.

---

## DDL (Schema das Tabelas)

```sql
-- Tabela de Clientes
CREATE TABLE IF NOT EXISTS clientes (
    id     INT,
    nome   STRING,
    email  STRING,
    cidade STRING
) USING DELTA LOCATION './delta-tables/clientes';

-- Tabela de Produtos
CREATE TABLE IF NOT EXISTS produtos (
    id        INT,
    nome      STRING,
    categoria STRING,
    preco     DOUBLE,
    estoque   INT
) USING DELTA LOCATION './delta-tables/produtos';

-- Tabela de Pedidos
CREATE TABLE IF NOT EXISTS pedidos (
    id          INT,
    cliente_id  INT,
    produto_id  INT,
    quantidade  INT,
    valor_total DOUBLE,
    data_pedido STRING,
    status      STRING
) USING DELTA LOCATION './delta-tables/pedidos';
```

## 1. Configuração do Ambiente

In [ ]:
import os
import sys
import shutil

# Garante que o Spark usa o Python do ambiente virtual
python_path = sys.executable
os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path
os.environ["HADOOP_HOME"] = "C:\\hadoop"
# Carrega hadoop.dll para suporte a operações de arquivo no Windows
os.environ["JAVA_TOOL_OPTIONS"] = "-Djava.library.path=C:\\hadoop\\bin"

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Diretório base para as tabelas Delta
BASE = os.path.abspath("./delta-tables").replace("\\", "/")

# Limpa tabelas anteriores para execução limpa
shutil.rmtree("./delta-tables", ignore_errors=True)

builder = (
    SparkSession.builder
    .appName("Delta Lake - Loja Virtual")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print(f"PySpark {spark.version} iniciado com Delta Lake!")
print(f"Tabelas serão salvas em: {BASE}")

## 2. Criação das Tabelas Delta (DDL + INSERT inicial)

In [ ]:
# DDL + INSERT - Clientes (usando CREATE TABLE AS SELECT FROM VALUES)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS clientes
    USING DELTA
    LOCATION '{BASE}/clientes'
    AS SELECT * FROM VALUES
        (1, 'Ana Souza',    'ana@email.com',    'Florianopolis'),
        (2, 'Bruno Lima',   'bruno@email.com',  'Criciuma'),
        (3, 'Carla Mendes', 'carla@email.com',  'Joinville'),
        (4, 'Diego Ramos',  'diego@email.com',  'Blumenau'),
        (5, 'Elena Costa',  'elena@email.com',  'Chapeco')
    AS t(id, nome, email, cidade)
""")

# DDL + INSERT - Produtos
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS produtos
    USING DELTA
    LOCATION '{BASE}/produtos'
    AS SELECT * FROM VALUES
        (1, 'Notebook Dell',    'Eletronicos', 3500.0, 50),
        (2, 'Mouse Logitech',   'Perifericos',   80.0, 200),
        (3, 'Teclado Mecanico', 'Perifericos',  150.0, 120),
        (4, 'Monitor 24pol',    'Eletronicos', 1200.0,  75),
        (5, 'Headset Gaming',   'Perifericos',  250.0,  90)
    AS t(id, nome, categoria, preco, estoque)
""")

# DDL + INSERT - Pedidos
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS pedidos
    USING DELTA
    LOCATION '{BASE}/pedidos'
    AS SELECT * FROM VALUES
        (1, 1, 1, 1, 3500.0, '2024-01-10', 'entregue'),
        (2, 2, 2, 2,  160.0, '2024-01-12', 'entregue'),
        (3, 3, 3, 1,  150.0, '2024-01-15', 'em_transito'),
        (4, 4, 4, 1, 1200.0, '2024-01-18', 'processando'),
        (5, 5, 5, 2,  500.0, '2024-01-20', 'processando')
    AS t(id, cliente_id, produto_id, quantidade, valor_total, data_pedido, status)
""")

print("=== Clientes ===")
spark.sql("SELECT * FROM clientes ORDER BY id").show()

print("=== Produtos ===")
spark.sql("SELECT * FROM produtos ORDER BY id").show()

print("=== Pedidos ===")
spark.sql("SELECT * FROM pedidos ORDER BY id").show()

## 3. INSERT — Inserindo Novos Registros

In [ ]:
# INSERT - Adiciona novos produtos
spark.sql("""
    INSERT INTO produtos VALUES
        (6, 'Webcam HD', 'Perifericos', 180.0, 60),
        (7, 'SSD 1TB',   'Eletronicos', 450.0, 40)
""")

# INSERT - Novo pedido
spark.sql("""
    INSERT INTO pedidos VALUES
        (6, 1, 6, 1, 180.0, '2024-01-25', 'processando')
""")

print("=== Produtos após INSERT ===")
spark.sql("SELECT * FROM produtos ORDER BY id").show()

print("=== Pedidos após INSERT ===")
spark.sql("SELECT * FROM pedidos ORDER BY id").show()

## 4. UPDATE — Atualizando Dados

In [ ]:
# UPDATE - Reajuste de 10% nos preços de Eletrônicos
spark.sql("""
    UPDATE produtos
    SET preco = preco * 1.10
    WHERE categoria = 'Eletronicos'
""")

# UPDATE - Marca pedidos entregues como finalizados
spark.sql("""
    UPDATE pedidos
    SET status = 'finalizado'
    WHERE status = 'entregue'
""")

print("=== Produtos após UPDATE (preco Eletronicos +10%) ===")
spark.sql("SELECT * FROM produtos ORDER BY id").show()

print("=== Pedidos após UPDATE (entregue -> finalizado) ===")
spark.sql("SELECT * FROM pedidos ORDER BY id").show()

## 5. DELETE — Removendo Dados

In [ ]:
# Zera estoque do produto 5 para demonstrar DELETE
spark.sql("UPDATE produtos SET estoque = 0 WHERE id = 5")

# DELETE - Remove produtos sem estoque
spark.sql("""
    DELETE FROM produtos
    WHERE estoque = 0
""")

print("=== Produtos após DELETE (sem estoque removidos) ===")
spark.sql("SELECT * FROM produtos ORDER BY id").show()

## 6. Consultas Analíticas

In [ ]:
# Total de vendas por status
print("=== Total de vendas por status ===")
spark.sql("""
    SELECT status,
           COUNT(*) AS qtd_pedidos,
           SUM(valor_total) AS total_vendas
    FROM pedidos
    GROUP BY status
    ORDER BY total_vendas DESC
""").show()

# Pedidos com nome do cliente e produto
print("=== Pedidos com detalhes ===")
spark.sql("""
    SELECT c.nome AS cliente, p.nome AS produto,
           ped.quantidade, ped.valor_total, ped.status
    FROM pedidos ped
    JOIN clientes c ON ped.cliente_id = c.id
    JOIN produtos p ON ped.produto_id = p.id
    ORDER BY ped.id
""").show()

# Detalhes da tabela Delta
print("=== Detalhes da tabela produtos ===")
spark.sql("DESCRIBE DETAIL produtos").select(
    "name", "format", "numFiles", "sizeInBytes"
).show(truncate=False)

spark.stop()
print("Sessão Spark encerrada.")